# Romania 2001: all-county offline reconciliation

This notebook reads **saved** Ministry checkpoints and reports where they agree. It does **not** contact the Wayback Machine, rank programs, or draw outcome curves. A finished download is not the same thing as a reconciled county, and a reconciled source is not authorization for a national outcome analysis.

Run the cells in order with the `sports_net` kernel. Cell 3 checks one county at a time and writes a name-free record before starting the next county. You can stop it and rerun it. Counties whose source files have not changed are skipped. Cell 4 then compares outgoing placements with incoming admissions.

## The views, in ordinary language

These are different Ministry report families recovered through the Wayback Machine. A page organized by school is still a Ministry report.

- **School Applicant View.** Applicants linked to an originating gymnasium, with examination and school-grade components. This is the view used to build an observed gymnasium peer pool. It does not show that every eighth grader applied.
- **School Result View.** A school-organized list of admitted and unassigned applicants, with destination labels where the page prints them. Its rows are not assumed to be the same people as the School Applicant View.
- **Resident applicant list.** County residents before allocation. This is a different list from the general county applicant list saved earlier.
- **Incoming lists.** Applicants from other counties, then the admitted and unassigned outcomes in the receiving county. A move here is an admissions route, not necessarily a change of residence.
- **County admitted, unassigned, and program views.** The original county lists, program directory, and occupancy table. `NEREPARTIZAT` means unassigned in this round. It does not mean the student was rejected from a named first choice.

The inventory's `B` unit is labelled **BUCURESTI si ILFOV** in the saved Ministry directory. That source label is kept. It is not split into a separate present-day Bucharest and Ilfov.

Printed name plus admission score is a provisional match. The archived personal identifier is masked. Duplicate keys stay visible. An unmatched row is unresolved, not unsuccessful.


In [1]:
# CELL 1 — Visible settings
from pathlib import Path

# Empty tuple: every county unit in the saved Ministry directory, in directory order.
# A short trial uses a tuple such as ("AB", "BN").
COUNTIES_TO_CHECK = ()

# True: skip a county whose saved check matches the current checkpoint sizes and times.
# False: start a new timestamped run folder and check every selected county again.
RESUME_UNCHANGED_COUNTIES = True

# Counties to check again even when resume would skip them. Example: ("BN",)
FORCE_RECHECK = ()

# None uses education_analysis/outputs/romania_2001_all_county_offline_reconciliation/
OUTPUT_ROOT = None

print("Counties:", "all 41 directory units" if not COUNTIES_TO_CHECK else ", ".join(COUNTIES_TO_CHECK))
print("Resume unchanged counties:", RESUME_UNCHANGED_COUNTIES)
print("Force recheck:", ", ".join(FORCE_RECHECK) if FORCE_RECHECK else "none")


Counties: all 41 directory units
Resume unchanged counties: True
Force recheck: none


In [2]:
# CELL 2 — Load the offline reconciliation code. No pages are read yet.
import importlib
import sys

EDUCATION_ROOT = None
for parent in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [parent, parent / "education_analysis",
                      parent / "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis"]:
        if (candidate / "code/EDUCATION_20261014_romania_all_county_offline_reconciliation.py").exists():
            EDUCATION_ROOT = candidate
            break
    if EDUCATION_ROOT is not None:
        break
if EDUCATION_ROOT is None:
    raise FileNotFoundError("Set the working directory so the education_analysis folder can be found.")
if str(EDUCATION_ROOT / "code") not in sys.path:
    sys.path.insert(0, str(EDUCATION_ROOT / "code"))

import EDUCATION_20261014_romania_all_county_offline_reconciliation as recon
importlib.reload(recon)
if OUTPUT_ROOT is None:
    OUTPUT_ROOT = EDUCATION_ROOT / "outputs/romania_2001_all_county_offline_reconciliation"
print("Code:", recon.__file__)
print("Output root:", OUTPUT_ROOT)
print("Private name ledger, not written to the repository:", recon.LEDGER_ROOT)


Code: /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis/code/EDUCATION_20261014_romania_all_county_offline_reconciliation.py
Output root: /Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE/3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis/outputs/romania_2001_all_county_offline_reconciliation
Private name ledger, not written to the repository: /Users/charleslevine/Desktop/VECTOR_temp/romania_2001_all_county_reconciliation_ledgers


## Cell 3 — Check each county

This cell hash-checks saved pages. It can take a long time because it opens every selected School Result View, resident list, and incoming list, and, where the files exist, the School Applicant View. It writes a county record, a progress line, and the summary tables **after each county**. Stop the cell if you need to. The next run resumes.

One county that stops does not stop the others. That county is left unmarked so a later run will try it again.


In [3]:
# CELL 3 — County pass. Rerun to resume. This does not use the network.
run_folder = recon.run_counties(
    COUNTIES_TO_CHECK,
    OUTPUT_ROOT,
    resume=RESUME_UNCHANGED_COUNTIES,
    force=FORCE_RECHECK,
)
print("Run folder:", run_folder)
print("Name-free report:", run_folder / "report.md")


2026-10-14 19:00:37 EDT | Started a new run folder: run_20261014T190037-0400
2026-10-14 19:00:37 EDT | Counties in this pass: 41. Each saved page is hash-checked. Nothing is downloaded.
2026-10-14 19:00:37 EDT | 1/41 ALBA (AB): starting.
2026-10-14 19:00:37 EDT | ALBA (AB): reading saved checkpoints. No archive requests.
2026-10-14 19:00:37 EDT | ALBA (AB): checking School Result View.
2026-10-14 19:00:37 EDT | ALBA (AB): checking resident applicant list.
2026-10-14 19:00:37 EDT | ALBA (AB): checking incoming applicant list.
2026-10-14 19:00:37 EDT | ALBA (AB): checking incoming admitted list.
2026-10-14 19:00:37 EDT | ALBA (AB): checking incoming unassigned list.
VERIFIED AB: 168 gymnasium pages, 3041 applicant rows
2026-10-14 19:00:37 EDT | ALBA (AB): ALBA (AB). School Result View: 168 verified school pages plus 1 directory page (page target 169, which includes the directory), 3042 printed rows, 0 unresolved URL addresses, 0 malformed or missing checkpoints. School Applicant View: 16

## Cell 4 — Cross-county routes

This compares two independent views. The School Result View in the origin county prints a destination school. The incoming admitted list in the receiving county prints an origin school. The comparison first asks whether the row counts agree, then whether printed name and admission score support a unique match.

Run Cell 3 first in this session, or leave `run_folder` unset and this cell uses the latest run folder. Names are read from the private ledger on the Desktop. They are not printed and they are not written into the repository.


In [4]:
# CELL 4 — Outgoing placements versus incoming admissions. No archive requests.
if "run_folder" not in dir():
    run_folder = recon.latest_run(OUTPUT_ROOT)
flow_counts = recon.cross_county_routes(run_folder)
print("Pair rows:", len(flow_counts))
print("Saved table:", run_folder / "cross_county_flow_counts.csv")
if len(flow_counts):
    disagree = flow_counts.loc[~flow_counts["aggregate_counts_agree"]]
    print("Pairs whose aggregate counts differ:", len(disagree))
    display_columns = [
        "origin_county", "destination_county",
        "school_result_outgoing_rows", "incoming_admitted_rows",
        "aggregate_counts_agree", "unique_name_score_keys_on_both_sides",
        "keys_only_on_school_result", "keys_only_on_incoming_admitted",
        "ambiguous_keys", "comparison_scope",
    ]
    display(flow_counts[display_columns].head(30))


2026-10-14 19:01:52 EDT | Cross-county routes: 41 counties in run_20261014T190037-0400. No archive requests.
2026-10-14 19:01:52 EDT | ALBA (AB): outgoing suffix rows 77, incoming admitted suffix rows 19.
2026-10-14 19:01:52 EDT | ARGES (AG): outgoing suffix rows 53, incoming admitted suffix rows 171.
2026-10-14 19:01:52 EDT | ARAD (AR): outgoing suffix rows 43, incoming admitted suffix rows 71.
2026-10-14 19:01:52 EDT | BUCURESTI si ILFOV (B): outgoing suffix rows 53, incoming admitted suffix rows 1317.
2026-10-14 19:01:52 EDT | BACAU (BC): outgoing suffix rows 153, incoming admitted suffix rows 126.
2026-10-14 19:01:52 EDT | BIHOR (BH): outgoing suffix rows 21, incoming admitted suffix rows 61.
2026-10-14 19:01:52 EDT | BISTRITA-NASAUD (BN): outgoing suffix rows 92, incoming admitted suffix rows 31.
2026-10-14 19:01:52 EDT | BRAILA (BR): outgoing suffix rows 65, incoming admitted suffix rows 57.
2026-10-14 19:01:52 EDT | BOTOSANI (BT): outgoing suffix rows 100, incoming admitted suff

,origin_county,destination_county,school_result_outgoing_rows,incoming_admitted_rows,aggregate_counts_agree,unique_name_score_keys_on_both_sides,keys_only_on_school_result,keys_only_on_incoming_admitted,ambiguous_keys,comparison_scope
0,AB,AR,1,1,True,1,0,0,0,both sides verified for known pages
1,AB,B,1,1,True,1,0,0,0,both sides verified for known pages
2,AB,BC,1,1,True,1,0,0,0,both sides verified for known pages
3,AB,BH,8,8,True,8,0,0,0,both sides verified for known pages
4,AB,BN,1,1,True,1,0,0,0,both sides verified for known pages
5,AB,CJ,15,15,True,15,0,0,0,both sides verified for known pages
6,AB,HD,18,18,True,18,0,0,0,both sides verified for known pages
7,AB,MS,10,10,True,10,0,0,0,both sides verified for known pages
8,AB,SB,19,19,True,19,0,0,0,both sides verified for known pages
9,AB,TM,3,3,True,3,0,0,0,both sides verified for known pages


## Cell 5 — Read the county summaries

The table is name-free. `plain_summary` is the sentence to read first. `next_source_question` is the specific gap, when one remains. A county with every known URL address saved can still be missing the School Applicant View, which is a different webpage family.


In [5]:
# CELL 5 — Show the saved county summaries.
if "run_folder" not in dir():
    run_folder = recon.latest_run(OUTPUT_ROOT)
summary = recon.summary_frame(run_folder)
columns = [
    "county", "county_name",
    "school_applicant_pages_saved", "school_applicant_pages_expected",
    "school_result_pages_verified", "school_result_directory_pages", "school_result_pages_expected",
    "school_result_unresolved_urls",
    "resident_pages_saved", "resident_pages_projected",
    "outgoing_rows_with_other_county_suffix",
    "original_placement_shortfall", "remaining_placement_shortfall",
    "next_source_question",
]
print(f"{len(summary)} counties in {run_folder.name}")
summary[columns]


41 counties in run_20261014T190037-0400


,county,county_name,school_applicant_pages_saved,school_applicant_pages_expected,school_result_pages_verified,school_result_directory_pages,school_result_pages_expected,school_result_unresolved_urls,resident_pages_saved,resident_pages_projected,outgoing_rows_with_other_county_suffix,original_placement_shortfall,remaining_placement_shortfall,next_source_question
0,AB,ALBA,168,168,168,1,169,0,7,7,77,0,0,Saved page checks finished without a new missi...
1,AG,ARGES,0,243,243,1,244,0,7,13,53,4846,0,School Applicant View pages saved: 0 of 243. A...
2,AR,ARAD,136,136,136,1,137,0,7,7,43,0,0,Saved page checks finished without a new missi...
3,B,BUCURESTI si ILFOV,289,289,289,1,290,0,6,45,53,18969,0,Resident applicant list pages saved: 6 of 45. ...
4,BC,BACAU,0,292,292,1,293,0,10,13,153,0,0,School Applicant View pages saved: 0 of 292. A...
5,BH,BIHOR,0,248,248,1,249,0,9,10,21,0,0,School Applicant View pages saved: 0 of 248. A...
6,BN,BISTRITA-NASAUD,0,146,145,1,147,1,6,6,92,1342,0,1 School Result View checkpoints are malformed...
7,BR,BRAILA,0,119,119,1,120,0,6,6,65,1098,0,School Applicant View pages saved: 0 of 119. A...
8,BT,BOTOSANI,0,214,214,1,215,0,8,8,100,0,0,School Applicant View pages saved: 0 of 214. A...
9,BV,BRASOV,0,142,142,1,143,0,9,10,71,439,0,School Applicant View pages saved: 0 of 142. A...


## What a count does and does not establish

- **School Applicant View saved / listed schools.** File coverage against the origin school directory. Person comparison with the School Result View runs only when every listed school page is present. A missing page stays unresolved.
- **School Result View verified / target.** `school_result_pages_verified` counts school pages. `school_result_directory_pages` counts the directory page. The target includes both, so a complete county often shows one more page in the target than in the school-page count. Unresolved URL addresses stay open. BN and SV each had one unresolved School Result View URL address in the October 14 inventory. This notebook does not close them.
- **Resident applicant list.** An incomplete list is a saved subset. People on missing pages are not dropped from the county and are not counted as unsuccessful.
- **Incoming row comparisons.** Printed name and admission score, with repeats kept. An incoming applicant row without an outcome row is unresolved.
- **Outgoing suffix.** A School Result View destination label that ends in another county code. It is an observed admissions route.
- **Placement shortfall.** Occupancy admitted total minus the original county admitted list. Program-specific recovery can reduce the **remaining** shortfall without forcing the original gap to zero.
- **Unique name-and-score key.** One printed name and admission score occurring once on each side of a county pair. Aggregate agreement does not by itself establish that link. The masked personal identifier means even a unique key is provisional.
- **Next source question.** The concrete missing evidence. It is not a label that the county is ready for a national outcome model.

Full sentences for every finished county are in `report.md` inside the run folder. Pair counts are in `cross_county_flow_counts.csv`.
